# Video Studio — `sdxl` on Colab (manual backup)

Kaggle is the studio's automated GPU provider. Use this notebook only when Kaggle has no GPU free.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Stay on the page.
2. When it finishes, the browser downloads **`vs-result-sdxl.zip`**. Give that file to Claude
   (upload it to the session or commit it to the branch); Claude runs `vs-colab import`.
3. The last cell releases the GPU so your Colab allowance is not wasted.

Colab rules: run it yourself in this notebook UI; no SSH, remote desktop or web UIs; one account;
no deepfakes — lip sync only on faces you own, AI-generated people, or people who agreed in writing.
Free Colab limits are not published and change; a run can be stopped if you are idle.

Same code as the Kaggle job (packed by `vs-colab build` from `gpu-jobs/`); models are downloaded
from their official Hugging Face / GitHub sources at pinned revisions.

In [ ]:
# 1. GPU check + map the Kaggle paths the job uses onto /content
import os, subprocess, pathlib
r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
assert r.returncode == 0, 'No GPU: Runtime > Change runtime type > T4 GPU, then Run all again'
print(r.stdout)
for d in ('/content/kaggle/working', '/content/kaggle/temp'):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
if not os.path.exists('/kaggle'):
    os.symlink('/content/kaggle', '/kaggle')


In [ ]:
# 3. the studio job (generated — do not edit; rebuild with vs-colab build)
JOB = '# --- packed by vs-gpu: inputs, params, shared runtime (gpu-jobs/_common.py), then the job ---\nimport base64, json, os, pathlib, time\n_T0 = time.time()\nIN = pathlib.Path("/kaggle/working/in"); OUT = pathlib.Path("/kaggle/working/out")\nIN.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)\nPARAMS = json.loads(\'{"model": "sdxl", "items": [{"id": "kid-reading", "prompt": "portrait of a cheerful 7-year-old Egyptian girl reading a colourful picture book on a cosy sofa, face clearly visible, looking at camera, warm afternoon window light, storybook illustration style", "width": 720, "height": 1280, "seed": 11}, {"id": "courier", "prompt": "a young courier on an orange scooter on a sunny Cairo street at sunrise, cinematic photo, 35mm, shallow depth of field", "width": 1280, "height": 720, "seed": 7}, {"id": "presenter", "prompt": "photo of a friendly young Arab woman news presenter facing the camera, head and shoulders, studio lighting, neutral background, sharp focus", "width": 1024, "height": 1024, "seed": 21}], "item_budget_s": 240, "max_job_s": 1500, "_need_gpu": true, "_need_internet": true, "parallel": false}\')\nfor _n, _b in json.loads(\'{}\').items():\n    (IN / _n).write_bytes(base64.b64decode(_b))\n_COMMON_B64 = \'IyBTaGFyZWQgcnVudGltZSBmb3IgZXZlcnkgc3R1ZGlvIEdQVSBqb2IgKHBhY2tlZCBieSB2cy1ncHU7IHJ1bnMgaW5zaWRlIHRoZSBLYWdnbGUga2VybmVsIGFuZCBpdHMKIyBwZXItR1BVIGNoaWxkIHByb2Nlc3NlcykuIFByb3ZpZGVzOiByZXBvcnQoKSwgZ3B1X2luZm8oKSwgaXNfdDQoKSwgcGlja19kdHlwZSgpLCBzaGFyZCgpLCBsb2FkX3BpcGUoKS4KIyBLYWdnbGUgZnJlZSB0aWVyID0gMnggVGVzbGEgVDQgKHNtNzUsIDE0LjYgR0IgZWFjaCk6IGZhc3QgZnAxNiB0ZW5zb3IgY29yZXMsIGJmMTYgb25seSBlbXVsYXRlZC4KaW1wb3J0IGpzb24sIG9zLCBwYXRobGliLCBzdWJwcm9jZXNzLCBzeXMsIHRpbWUKClNIQVJEID0gaW50KG9zLmVudmlyb24uZ2V0KCJWU19TSEFSRCIsICItMSIpKSAgICAgICAgICAjID49IDAgaW5zaWRlIGEgcGVyLUdQVSBjaGlsZCBwcm9jZXNzCgoKZGVmIHJlcG9ydCgqKmt3KToKICAgIHAgPSBPVVQgLyAoInJlcG9ydC5qc29uIiBpZiBTSEFSRCA8IDAgZWxzZSBmInJlcG9ydC5zaGFyZHtTSEFSRH0uanNvbiIpCiAgICBkID0ganNvbi5sb2FkcyhwLnJlYWRfdGV4dCgpKSBpZiBwLmV4aXN0cygpIGVsc2Uge30KICAgIGQudXBkYXRlKGt3KTsgZFsiZWxhcHNlZF9zIl0gPSByb3VuZCh0aW1lLnRpbWUoKSAtIF9UMCwgMSk7IHAud3JpdGVfdGV4dChqc29uLmR1bXBzKGQsIGluZGVudD0yKSkKCgpkZWYgX3ZyYW1fcGVhaygpOiAgICAgICAgICAgIyBiZW5jaG1hcms6IHBlYWsgVlJBTSBwZXIgdmlzaWJsZSBHUFUsIHdyaXR0ZW4gYXQgZXhpdCBvZiBldmVyeSAoY2hpbGQpIHByb2Nlc3MKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICByZXBvcnQoKip7InZyYW1fcGVha19nYiI6IFtyb3VuZCh0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKGkpIC8gMioqMzAsIDIpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkpXX0pCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKCgpkZWYgX3dhdGNoZG9nKGxpbWl0X3MpOiAgICAjIGhhcmQgam9iIGJ1ZGdldDogbmV2ZXIgbGV0IGEgc3R1Y2sgam9iIGJ1cm4gR1BVIHF1b3RhIChLYWdnbGUgYWxsb3dzIDEyIGgpCiAgICBpbXBvcnQgdGhyZWFkaW5nCiAgICBkZWYgZmlyZSgpOgogICAgICAgIHJlcG9ydChlcnJvcj1mImpvYiBleGNlZWRlZCBpdHMgYnVkZ2V0IG9mIHtsaW1pdF9zfSBzIGFuZCB3YXMgc3RvcHBlZCAocmFpc2UgbWF4X2pvYl9zIG9ubHkgaWYgbmVlZGVkKSIsCiAgICAgICAgICAgICAgIHRpbWVkX291dD1UcnVlKQogICAgICAgIF92cmFtX3BlYWsoKTsgb3MuX2V4aXQoNSkKICAgIHQgPSB0aHJlYWRpbmcuVGltZXIobGltaXRfcywgZmlyZSk7IHQuZGFlbW9uID0gVHJ1ZTsgdC5zdGFydCgpCgoKaW1wb3J0IGF0ZXhpdAphdGV4aXQucmVnaXN0ZXIoX3ZyYW1fcGVhaykKaWYgU0hBUkQgPCAwIGFuZCBub3Qgb3MuZW52aXJvbi5nZXQoIlZTX0ZBS0VfR1BVUyIpOgogICAgX3dhdGNoZG9nKGludChQQVJBTVMuZ2V0KCJtYXhfam9iX3MiLCAzNjAwKSkpCgoKZGVmIHN0ZXBfZ3VhcmQodG90YWxfc3RlcHMsIGJ1ZGdldF9zLCBsYWJlbD0iaXRlbSIsIHdlaWdodHM9Tm9uZSk6CiAgICAiIiJkaWZmdXNlcnMgY2FsbGJhY2tfb25fc3RlcF9lbmQ6IHN0b3Agb24gTmFOL2luZiBsYXRlbnRzIGFuZCB3aGVuIHRoZSBwcm9qZWN0ZWQgdGltZSBvZiB0aGlzIGl0ZW0gZXhjZWVkcwogICAgYnVkZ2V0X3MgKGVhcmx5IGZhaWx1cmUgZGV0ZWN0aW9uLCBtZWFzdXJlZCBwZXIgc3RlcCkuIHdlaWdodHMgPSByZWxhdGl2ZSBjb3N0IHBlciBzdGVwICgyIHdpdGggQ0ZHLCAxCiAgICB3aXRob3V0KSBzbyB0aGUgcHJvamVjdGlvbiBzdGF5cyByaWdodCB3aGVuIENGRyBpcyBzd2l0Y2hlZCBvZmYgcGFydC13YXkuIFJhaXNlcyBGbG9hdGluZ1BvaW50RXJyb3IgLwogICAgVGltZW91dEVycm9yLiIiIgogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgdyA9IGxpc3Qod2VpZ2h0cyBvciBbMV0gKiB0b3RhbF9zdGVwcykKICAgIHN0YXRzID0geyJzdGVwcyI6IDAsICJkZW5vaXNlX3MiOiAwLjB9CiAgICBkZWYgY2IocGlwZSwgc3RlcCwgdHMsIGt3KToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBsYXQgPSBrdy5nZXQoImxhdGVudHMiKQogICAgICAgIGlmIGxhdCBpcyBub3QgTm9uZSBhbmQgbm90IHRvcmNoLmlzZmluaXRlKGxhdCkuYWxsKCk6CiAgICAgICAgICAgIHJhaXNlIEZsb2F0aW5nUG9pbnRFcnJvcihmIntsYWJlbH06IG5vbi1maW5pdGUgbGF0ZW50cyBhdCBzdGVwIHtzdGVwfSIpCiAgICAgICAgZWwgPSB0aW1lLnRpbWUoKSAtIHQwCiAgICAgICAgcHJvaiA9IGVsIC8gc3VtKHdbOnN0ZXAgKyAxXSkgKiBzdW0odykKICAgICAgICBpZiBzdGVwID49IDEgYW5kIHByb2ogPiBidWRnZXRfczoKICAgICAgICAgICAgcmFpc2UgVGltZW91dEVycm9yKGYie2xhYmVsfToge2VsIC8gKHN0ZXAgKyAxKTouMWZ9IHMvc3RlcCwgcHJvamVjdGVkIHtwcm9qOi4wZn0gcyA+IGJ1ZGdldCB7YnVkZ2V0X3N9IHMiKQogICAgICAgIHN0YXRzLnVwZGF0ZShzdGVwcz1zdGVwICsgMSwgZGVub2lzZV9zPXJvdW5kKGVsLCAxKSkKICAgICAgICByZXR1cm4ga3cKICAgIGNiLnN0YXRzID0gc3RhdHMKICAgIHJldHVybiBjYgoKCmRlZiBncHVfaW5mbygpOgogICAgaWYgb3MuZW52aXJvbi5nZXQoIlZTX0ZBS0VfR1BVUyIpOiAgICAgICAjIENQVSBzZWxmLXRlc3Qgb2YgcGFja2luZyBhbmQgZmFuLW91dCAodGVzdHMvdGVzdF9ncHVfcnVudGltZS5zaCkKICAgICAgICByZXR1cm4geyJuIjogaW50KG9zLmVudmlyb25bIlZTX0ZBS0VfR1BVUyJdKSwgIm5hbWUiOiAiZmFrZSIsICJ2cmFtX2diIjogMTQuNiwgImNjIjogNy41fQogICAgaW1wb3J0IHRvcmNoCiAgICBuID0gdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKQogICAgaWYgbm90IG46CiAgICAgICAgcmV0dXJuIHsibiI6IDB9CiAgICBwID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoMCkKICAgIHJldHVybiB7Im4iOiBuLCAibmFtZSI6IHAubmFtZSwgInZyYW1fZ2IiOiByb3VuZChwLnRvdGFsX21lbW9yeSAvIDIqKjMwLCAxKSwgImNjIjogcC5tYWpvciArIHAubWlub3IgLyAxMH0KCgpkZWYgaXNfdDQoKToKICAgIGcgPSBncHVfaW5mbygpCiAgICByZXR1cm4gZy5nZXQoIm4iLCAwKSA+IDAgYW5kIGdbImNjIl0gPCA4LjAKCgpkZWYgcGlja19kdHlwZShmcDE2X3NhZmUpOgogICAgIiIiZnAxNiBvbiBUNCBmb3IgbW9kZWxzIGtub3duIHRvIGJlIGZwMTYtc2FmZSAoNC04eCBmYXN0ZXIgdGhhbiBlbXVsYXRlZCBiZjE2KTsgYmYxNiBvdGhlcndpc2UuCiAgICBQQVJBTVNbImR0eXBlIl0gPSBmcDE2IHwgYmYxNiB8IGZwMzIgb3ZlcnJpZGVzLiIiIgogICAgaW1wb3J0IHRvcmNoCiAgICB3YW50ID0gUEFSQU1TLmdldCgiZHR5cGUiLCAiYXV0byIpCiAgICBpZiB3YW50IGluICgiZnAxNiIsICJiZjE2IiwgImZwMzIiKToKICAgICAgICByZXR1cm4geyJmcDE2IjogdG9yY2guZmxvYXQxNiwgImJmMTYiOiB0b3JjaC5iZmxvYXQxNiwgImZwMzIiOiB0b3JjaC5mbG9hdDMyfVt3YW50XQogICAgaWYgaXNfdDQoKToKICAgICAgICByZXR1cm4gdG9yY2guZmxvYXQxNiBpZiBmcDE2X3NhZmUgZWxzZSB0b3JjaC5iZmxvYXQxNgogICAgcmV0dXJuIHRvcmNoLmJmbG9hdDE2CgoKY2xhc3MgX0RvbmUoRXhjZXB0aW9uKToKICAgIHBhc3MKCgpkZWYgc2hhcmQoaXRlbXMpOgogICAgIiIiU21hbGwgbW9kZWxzOiBvbmUgcHJvY2VzcyBwZXIgR1BVLCBpdGVtcyBzcGxpdCByb3VuZC1yb2JpbiAoMnggdGhyb3VnaHB1dCBvbiAyeCBUNCkuCiAgICBSZXR1cm5zIHRoaXMgcHJvY2VzcydzIGl0ZW1zOyBpbiB0aGUgcGFyZW50IGl0IHdhaXRzIGZvciB0aGUgY2hpbGRyZW4sIG1lcmdlcyByZXBvcnRzIGFuZCBzdG9wcyB0aGUgam9iLiIiIgogICAgbiA9IGdwdV9pbmZvKCkuZ2V0KCJuIiwgMCkKICAgIGlmIFNIQVJEID49IDA6CiAgICAgICAgcmV0dXJuIGl0ZW1zW1NIQVJEOjppbnQob3MuZW52aXJvblsiVlNfTlNIQVJEIl0pXQogICAgaWYgbiA8IDIgb3IgbGVuKGl0ZW1zKSA8IDIgb3Igbm90IFBBUkFNUy5nZXQoInBhcmFsbGVsIiwgVHJ1ZSk6CiAgICAgICAgcmV0dXJuIGl0ZW1zCiAgICB0bXAgPSBwYXRobGliLlBhdGgoIi9rYWdnbGUvdGVtcCIpOyB0bXAubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgKHRtcCAvICJ2c19wYXJhbXMuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhQQVJBTVMpKQogICAgKHRtcCAvICJ2c19jaGlsZC5weSIpLndyaXRlX3RleHQoCiAgICAgICAgImltcG9ydCBiYXNlNjQsIGpzb24sIHBhdGhsaWIsIHRpbWVcbl9UMCA9IHRpbWUudGltZSgpXG4iCiAgICAgICAgIklOID0gcGF0aGxpYi5QYXRoKCcva2FnZ2xlL3dvcmtpbmcvaW4nKTsgT1VUID0gcGF0aGxpYi5QYXRoKCcva2FnZ2xlL3dvcmtpbmcvb3V0JylcbiIKICAgICAgICAiUEFSQU1TID0ganNvbi5sb2FkKG9wZW4oJy9rYWdnbGUvdGVtcC92c19wYXJhbXMuanNvbicpKVxuIgogICAgICAgIGYiX0NPTU1PTl9CNjQgPSB7X0NPTU1PTl9CNjQhcn1cbl9KT0JfQjY0ID0ge19KT0JfQjY0IXJ9XG4iCiAgICAgICAgImV4ZWMoYmFzZTY0LmI2NGRlY29kZShfQ09NTU9OX0I2NCkuZGVjb2RlKCkpXG4iCiAgICAgICAgImV4ZWMoY29tcGlsZShiYXNlNjQuYjY0ZGVjb2RlKF9KT0JfQjY0KS5kZWNvZGUoKSwgJ2pvYi5weScsICdleGVjJykpXG4iKQogICAgcHJvY3MgPSBbc3VicHJvY2Vzcy5Qb3Blbihbc3lzLmV4ZWN1dGFibGUsIHN0cih0bXAgLyAidnNfY2hpbGQucHkiKV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGVudj17Kipvcy5lbnZpcm9uLCAiVlNfU0hBUkQiOiBzdHIoaSksICJWU19OU0hBUkQiOiBzdHIobiksICJDVURBX1ZJU0lCTEVfREVWSUNFUyI6IHN0cihpKX0pCiAgICAgICAgICAgICBmb3IgaSBpbiByYW5nZShuKV0KICAgIGNvZGVzID0gW3Aud2FpdCgpIGZvciBwIGluIHByb2NzXQogICAgc2hhcmRzID0gW10KICAgIGZvciBpIGluIHJhbmdlKG4pOgogICAgICAgIGYgPSBPVVQgLyBmInJlcG9ydC5zaGFyZHtpfS5qc29uIgogICAgICAgIGlmIGYuZXhpc3RzKCk6CiAgICAgICAgICAgIHNoYXJkcy5hcHBlbmQoanNvbi5sb2FkcyhmLnJlYWRfdGV4dCgpKSk7IGYudW5saW5rKCkKICAgIHJlcG9ydChwYXJhbGxlbF9ncHVzPW4sIHNoYXJkcz1zaGFyZHMsIHNoYXJkX2V4aXRfY29kZXM9Y29kZXMpCiAgICBpZiBjb2RlcyBhbmQgYWxsKGNvZGVzKToKICAgICAgICByZXBvcnQoZXJyb3I9ZiJldmVyeSBHUFUgd29ya2VyIGZhaWxlZDogZXhpdCBjb2RlcyB7Y29kZXN9IikKICAgIGVsaWYgYW55KGNvZGVzKTogICAgICAgICAgICAgICAgICAgICAgICAgICMgcGFydGlhbCByZXN1bHQ6IGtlZXAgd2hhdCB3YXMgbWFkZSwgdnMtcHJvZHVjZSBmYWxscyBiYWNrIGZvciB0aGUgcmVzdAogICAgICAgIHJlcG9ydCh3YXJuaW5nPWYiYSBHUFUgd29ya2VyIGZhaWxlZDogZXhpdCBjb2RlcyB7Y29kZXN9IikKICAgIHJhaXNlIF9Eb25lKCkKCgpkZWYgbG9hZF9waXBlKGNscywgcmVwbywgcmV2aXNpb24sIGR0eXBlLCBmcDhfb2s9VHJ1ZSwgcHJlZmVyPU5vbmUsICoqa3cpOgogICAgIiIiQmlnIGRpZmZ1c2lvbiBtb2RlbHMgb24gMnggVDQsIGJlc3Qgc3RyYXRlZ3kgZmlyc3Q6CiAgICAgICAxKSBkZXZpY2VfbWFwPSdiYWxhbmNlZCcg4oCUIHRleHQgZW5jb2RlciBvbiBvbmUgVDQsIHRyYW5zZm9ybWVyICsgVkFFIG9uIHRoZSBvdGhlciAobm8gQ1BVIHJvdW5kLXRyaXBzKQogICAgICAgMikgZnA4IHdlaWdodCBzdG9yYWdlIGZvciB0aGUgdHJhbnNmb3JtZXIgKyBtb2RlbCBDUFUgb2ZmbG9hZCBvbiBvbmUgVDQKICAgICAgIDMpIHNlcXVlbnRpYWwgQ1BVIG9mZmxvYWQgKHNsb3dlc3QsIGFsd2F5cyBmaXRzKQogICAgUEFSQU1TWyJwbGFjZW1lbnQiXSA9IGJhbGFuY2VkIHwgb2ZmbG9hZCB8IHNlcXVlbnRpYWwgZm9yY2VzIG9uZS4iIiIKICAgIGltcG9ydCB0b3JjaAogICAgZm9yY2UgPSBwcmVmZXIgb3IgUEFSQU1TLmdldCgicGxhY2VtZW50IikKICAgIGVycm9ycyA9IFtdCiAgICBpZiAoZm9yY2UgaW4gKE5vbmUsICJiYWxhbmNlZCIpKSBhbmQgZ3B1X2luZm8oKS5nZXQoIm4iLCAwKSA+PSAyOgogICAgICAgIHRyeToKICAgICAgICAgICAgcGlwZSA9IGNscy5mcm9tX3ByZXRyYWluZWQocmVwbywgcmV2aXNpb249cmV2aXNpb24sIHRvcmNoX2R0eXBlPWR0eXBlLCBkZXZpY2VfbWFwPSJiYWxhbmNlZCIsICoqa3cpCiAgICAgICAgICAgIGZwOCA9IGZwOF9vayBhbmQgZ3B1X2luZm8oKVsidnJhbV9nYiJdIDwgMjAgYW5kIGhhc2F0dHIocGlwZSwgInRyYW5zZm9ybWVyIikKICAgICAgICAgICAgaWYgZnA4OiAgICMgbWVhc3VyZWQ6IFotSW1hZ2UncyA2QiB0cmFuc2Zvcm1lciBhbG9uZSAoMTIgR0IgYmYxNikgKyBhdHRlbnRpb24gb3ZlcmZsb3dzIGEgMTQuNiBHQiBUNAogICAgICAgICAgICAgICAgcGlwZS50cmFuc2Zvcm1lci5lbmFibGVfbGF5ZXJ3aXNlX2Nhc3Rpbmcoc3RvcmFnZV9kdHlwZT10b3JjaC5mbG9hdDhfZTRtM2ZuLCBjb21wdXRlX2R0eXBlPWR0eXBlKQogICAgICAgICAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCiAgICAgICAgICAgIHJlcG9ydChwbGFjZW1lbnQ9ImJhbGFuY2VkICgyIEdQVXMpIiArICgiICsgZnA4IHRyYW5zZm9ybWVyIHdlaWdodHMiIGlmIGZwOCBlbHNlICIiKSkKICAgICAgICAgICAgcmV0dXJuIHBpcGUKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJiYWxhbmNlZDoge3R5cGUoZSkuX19uYW1lX199OiB7c3RyKGUpWzoyMDBdfSIpCiAgICBpZiBmb3JjZSBpbiAoTm9uZSwgIm9mZmxvYWQiKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHBpcGUgPSBjbHMuZnJvbV9wcmV0cmFpbmVkKHJlcG8sIHJldmlzaW9uPXJldmlzaW9uLCB0b3JjaF9kdHlwZT1kdHlwZSwgKiprdykKICAgICAgICAgICAgaWYgZnA4X29rIGFuZCBncHVfaW5mbygpWyJ2cmFtX2diIl0gPCAyMCBhbmQgaGFzYXR0cihwaXBlLCAidHJhbnNmb3JtZXIiKToKICAgICAgICAgICAgICAgIHBpcGUudHJhbnNmb3JtZXIuZW5hYmxlX2xheWVyd2lzZV9jYXN0aW5nKHN0b3JhZ2VfZHR5cGU9dG9yY2guZmxvYXQ4X2U0bTNmbiwgY29tcHV0ZV9kdHlwZT1kdHlwZSkKICAgICAgICAgICAgcGlwZS5lbmFibGVfbW9kZWxfY3B1X29mZmxvYWQoZ3B1X2lkPTApCiAgICAgICAgICAgIHJlcG9ydChwbGFjZW1lbnQ9ImZwOCB0cmFuc2Zvcm1lciArIG1vZGVsIENQVSBvZmZsb2FkICgxIEdQVSkiLCBwbGFjZW1lbnRfZXJyb3JzPWVycm9ycykKICAgICAgICAgICAgcmV0dXJuIHBpcGUKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJvZmZsb2FkOiB7dHlwZShlKS5fX25hbWVfX306IHtzdHIoZSlbOjIwMF19IikKICAgIHBpcGUgPSBjbHMuZnJvbV9wcmV0cmFpbmVkKHJlcG8sIHJldmlzaW9uPXJldmlzaW9uLCB0b3JjaF9kdHlwZT1kdHlwZSwgKiprdykKICAgIHBpcGUuZW5hYmxlX3NlcXVlbnRpYWxfY3B1X29mZmxvYWQoKQogICAgcmVwb3J0KHBsYWNlbWVudD0ic2VxdWVudGlhbCBDUFUgb2ZmbG9hZCIsIHBsYWNlbWVudF9lcnJvcnM9ZXJyb3JzKQogICAgcmV0dXJuIHBpcGUKCgpkZWYgdmFlX3RpbGluZyhwaXBlKToKICAgIGZvciBuYW1lIGluICgidmFlIiwpOgogICAgICAgIG0gPSBnZXRhdHRyKHBpcGUsIG5hbWUsIE5vbmUpCiAgICAgICAgZm9yIGZuIGluICgiZW5hYmxlX3RpbGluZyIsICJlbmFibGVfc2xpY2luZyIpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBnZXRhdHRyKG0sIGZuKSgpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICBwYXNzCgoKZGVmIGxvb2tzX2Jyb2tlbihhcnIpOgogICAgIiIiTmFOL2JsYWNrIG91dHB1dCBjaGVjayAoZnAxNiBvdmVyZmxvdyBzaG93cyB1cCBhcyBibGFjayBmcmFtZXMpLiIiIgogICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICBhID0gbnAuYXNhcnJheShhcnIsIGR0eXBlPW5wLmZsb2F0MzIpCiAgICByZXR1cm4gKG5vdCBucC5pc2Zpbml0ZShhKS5hbGwoKSkgb3IgZmxvYXQoYS5tZWFuKCkpIDwgMi4wCg==\'\n_JOB_B64 = \'IyBHUFUgam9iOiB0ZXh0LXRvLWltYWdlLCB0dW5lZCBmb3IgS2FnZ2xlJ3MgMnggVGVzbGEgVDQgKDE0LjYgR0IgZWFjaCwgc203NSkuCiMgUEFSQU1TOiB7Im1vZGVsIjogInNkeGwiIHwgInppbWFnZSIsICJpdGVtcyI6IFt7ImlkIiwicHJvbXB0IiwibmVnYXRpdmUiLCJ3aWR0aCIsImhlaWdodCIsInNlZWQifV0sICJzdGVwcyI6IE4sCiMgICAgICAgICAgIml0ZW1fYnVkZ2V0X3MiOiAyNDAsICJtYXhfam9iX3MiOiAzNjAwfQojICAgc2R4bCAgICBERUZBVUxUIG9uIFQ0LiBTRFhMIGJhc2UgMS4wIChPcGVuUkFJTCsrKSArIGZwMTYtZml4IFZBRSAoTUlUKTogbmF0aXZlIGZwMTYsIH43IEdCLCBvbmUgY29weSBwZXIgVDQgc28KIyAgICAgICAgICAgYm90aCBHUFVzIHdvcmsgaW4gcGFyYWxsZWwsIDI4IHN0ZXBzLiBSZWxpYWJsZSBvbiB0aGUgZnJlZSB0aWVyLgojICAgemltYWdlICBBRFZBTkNFRCAob3B0aW9uYWwpLiBaLUltYWdlLVR1cmJvIDZCIChBcGFjaGUtMi4wKTogMTIgR0IgdHJhbnNmb3JtZXIg4oCUIG9uIFQ0IGl0IG5lZWRzIGJvdGggR1BVcyArCiMgICAgICAgICAgIGZwOCB3ZWlnaHRzIChPT00gaW4gYmYxNiwgbWVhc3VyZWQpLiBGYWxscyBiYWNrIHRvIFNEWEwgYXV0b21hdGljYWxseSBvbiBhbnkgZmFpbHVyZS4KIyBFdmVyeSBpbWFnZSBpcyBjaGVja2VkIGZvciBOYU4vYmxhY2sgb3V0cHV0IChmcDE2IG92ZXJmbG93KSBhbmQgcmV0cmllZCBvbmNlIGluIGEgc2FmZXIgbW9kZS4KaW1wb3J0IG9zLCBzdWJwcm9jZXNzLCBzeXMKb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJQWVRPUkNIX0NVREFfQUxMT0NfQ09ORiIsICJleHBhbmRhYmxlX3NlZ21lbnRzOlRydWUiKQpzdWJwcm9jZXNzLnJ1bihbc3lzLmV4ZWN1dGFibGUsICItbSIsICJwaXAiLCAiaW5zdGFsbCIsICItcSIsICJkaWZmdXNlcnM9PTAuNDMuMCIsICJhY2NlbGVyYXRlIiwgInRyYW5zZm9ybWVycyIsCiAgICAgICAgICAgICAgICAic2FmZXRlbnNvcnMiXSwgY2hlY2s9VHJ1ZSkKaW1wb3J0IHRvcmNoCk1PREVMUyA9IHsiemltYWdlIjogKCJUb25neWktTUFJL1otSW1hZ2UtVHVyYm8iLCAiZjMzMjA3MmFhNzhiIiksCiAgICAgICAgICAic2R4bCI6ICgic3RhYmlsaXR5YWkvc3RhYmxlLWRpZmZ1c2lvbi14bC1iYXNlLTEuMCIsICI0NjIxNjU5ODQwMzAiKX0KbW9kZWwgPSBQQVJBTVMuZ2V0KCJtb2RlbCIpIG9yICgic2R4bCIgaWYgaXNfdDQoKSBlbHNlICJ6aW1hZ2UiKQoKCmRlZiBnZW5femltYWdlKGl0ZW1zKToKICAgIGZyb20gZGlmZnVzZXJzIGltcG9ydCBaSW1hZ2VQaXBlbGluZQogICAgdCA9IHRpbWUudGltZSgpCiAgICBkdHlwZSA9IHBpY2tfZHR5cGUoZnAxNl9zYWZlPUZhbHNlKSAgICAgICAgICAjIFotSW1hZ2UgaXMgdHJhaW5lZCBpbiBiZjE2OyBmcDE2IG9ubHkgaWYgUEFSQU1TIGR0eXBlPWZwMTYKICAgIHBpcGUgPSBsb2FkX3BpcGUoWkltYWdlUGlwZWxpbmUsICpNT0RFTFNbInppbWFnZSJdLCBkdHlwZSkKICAgIHZhZV90aWxpbmcocGlwZSkKICAgIHJlcG9ydChsb2FkX3M9cm91bmQodGltZS50aW1lKCkgLSB0LCAxKSwgZHR5cGU9c3RyKGR0eXBlKSwgbW9kZWw9InotaW1hZ2UtdHVyYm8iKQogICAgZG9uZSA9IFtdCiAgICBmb3IgaXQgaW4gaXRlbXM6CiAgICAgICAgcyA9IHRpbWUudGltZSgpCiAgICAgICAgZyA9IHRvcmNoLkdlbmVyYXRvcigiY3B1IikubWFudWFsX3NlZWQoaXQuZ2V0KCJzZWVkIiwgMCkpCiAgICAgICAgaW1nID0gcGlwZShwcm9tcHQ9aXRbInByb21wdCJdLCBoZWlnaHQ9aXQuZ2V0KCJoZWlnaHQiLCA3MjApLCB3aWR0aD1pdC5nZXQoIndpZHRoIiwgMTI4MCksCiAgICAgICAgICAgICAgICAgICBudW1faW5mZXJlbmNlX3N0ZXBzPVBBUkFNUy5nZXQoInN0ZXBzIiwgOCksIGd1aWRhbmNlX3NjYWxlPTAuMCwgZ2VuZXJhdG9yPWcpLmltYWdlc1swXQogICAgICAgIGlmIGxvb2tzX2Jyb2tlbihpbWcpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7aXRbJ2lkJ119OiBibGFjay9OYU4gb3V0cHV0IikKICAgICAgICBpbWcuc2F2ZShPVVQgLyBmIntpdFsnaWQnXX0ucG5nIik7IGRvbmUuYXBwZW5kKHsiaWQiOiBpdFsiaWQiXSwgInMiOiByb3VuZCh0aW1lLnRpbWUoKSAtIHMsIDEpfSkKICAgICAgICByZXBvcnQoaW1hZ2VzPWRvbmUpCgoKZGVmIGdlbl9zZHhsKGl0ZW1zKToKICAgIGZyb20gZGlmZnVzZXJzIGltcG9ydCBTdGFibGVEaWZmdXNpb25YTFBpcGVsaW5lLCBBdXRvZW5jb2RlcktMCiAgICBpdGVtcyA9IHNoYXJkKGl0ZW1zKSAgICAgICAgICAgICAgICAgICAgICAgICAjIG9uZSBTRFhMIGNvcHkgcGVyIFQ0CiAgICB0ID0gdGltZS50aW1lKCkKICAgIHZhZSA9IEF1dG9lbmNvZGVyS0wuZnJvbV9wcmV0cmFpbmVkKCJtYWRlYnlvbGxpbi9zZHhsLXZhZS1mcDE2LWZpeCIsIHJldmlzaW9uPSIyMDdiMTE2ZGFlNzAiLCB0b3JjaF9kdHlwZT10b3JjaC5mbG9hdDE2KQogICAgcGlwZSA9IFN0YWJsZURpZmZ1c2lvblhMUGlwZWxpbmUuZnJvbV9wcmV0cmFpbmVkKE1PREVMU1sic2R4bCJdWzBdLCByZXZpc2lvbj1NT0RFTFNbInNkeGwiXVsxXSwgdmFlPXZhZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0b3JjaF9kdHlwZT10b3JjaC5mbG9hdDE2LCB2YXJpYW50PSJmcDE2IikudG8oImN1ZGEiKQogICAgdmFlX3RpbGluZyhwaXBlKSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBkaWZmdXNlcnMgMC40MzogdGlsaW5nIGxpdmVzIG9uIHRoZSBWQUUKICAgIHJlcG9ydChsb2FkX3M9cm91bmQodGltZS50aW1lKCkgLSB0LCAxKSwgZHR5cGU9InRvcmNoLmZsb2F0MTYiLCBtb2RlbD0ic2R4bC1iYXNlLTEuMCIpCiAgICBkb25lID0gW10KICAgIG5lZyA9ICJibHVycnksIGxvd3JlcywgZGVmb3JtZWQsIGV4dHJhIGZpbmdlcnMsIHdhdGVybWFyaywgdGV4dCwgbG9nbyIKICAgIHN0ZXBzID0gUEFSQU1TLmdldCgic3RlcHNfc2R4bCIsIDI4KQogICAgZG9uZSwgZmFpbGVkID0gW10sIFtdCiAgICBmb3IgaXQgaW4gaXRlbXM6CiAgICAgICAgdywgaCA9IGl0LmdldCgid2lkdGgiLCAxMzQ0KSwgaXQuZ2V0KCJoZWlnaHQiLCA3NjgpCiAgICAgICAgdywgaCA9ICgxMzQ0LCA3NjgpIGlmIHcgPiBoIGVsc2UgKCg3NjgsIDEzNDQpIGlmIGggPiB3IGVsc2UgKDEwMjQsIDEwMjQpKSAgICMgU0RYTC1uYXRpdmUgYnVja2V0cwogICAgICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDIpOiAgICAgICAgICAgICAgICAgIyBvbmUgcmV0cnkgd2l0aCBhbm90aGVyIHNlZWQgb24gTmFOL2JsYWNrIG91dHB1dAogICAgICAgICAgICBzID0gdGltZS50aW1lKCkKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgaW1nID0gcGlwZShwcm9tcHQ9aXRbInByb21wdCJdLCBuZWdhdGl2ZV9wcm9tcHQ9aXQuZ2V0KCJuZWdhdGl2ZSIsIG5lZyksIHdpZHRoPXcsIGhlaWdodD1oLAogICAgICAgICAgICAgICAgICAgICAgICAgICBudW1faW5mZXJlbmNlX3N0ZXBzPXN0ZXBzLCBndWlkYW5jZV9zY2FsZT02LjAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIGNhbGxiYWNrX29uX3N0ZXBfZW5kPXN0ZXBfZ3VhcmQoc3RlcHMsIFBBUkFNUy5nZXQoIml0ZW1fYnVkZ2V0X3MiLCAyNDApLCBpdFsiaWQiXSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgIGdlbmVyYXRvcj10b3JjaC5HZW5lcmF0b3IoImN1ZGEiKS5tYW51YWxfc2VlZChpdC5nZXQoInNlZWQiLCAwKSArIGF0dGVtcHQpKS5pbWFnZXNbMF0KICAgICAgICAgICAgZXhjZXB0IChGbG9hdGluZ1BvaW50RXJyb3IsIFRpbWVvdXRFcnJvcikgYXMgZToKICAgICAgICAgICAgICAgIGZhaWxlZC5hcHBlbmQoeyJpZCI6IGl0WyJpZCJdLCAid2h5Ijogc3RyKGUpfSk7IGJyZWFrCiAgICAgICAgICAgIGlmIG5vdCBsb29rc19icm9rZW4oaW1nKToKICAgICAgICAgICAgICAgIGltZy5zYXZlKE9VVCAvIGYie2l0WydpZCddfS5wbmciKQogICAgICAgICAgICAgICAgZG9uZS5hcHBlbmQoeyJpZCI6IGl0WyJpZCJdLCAicyI6IHJvdW5kKHRpbWUudGltZSgpIC0gcywgMSksICJzaXplIjogZiJ7d314e2h9IiwgImF0dGVtcHQiOiBhdHRlbXB0ICsgMX0pCiAgICAgICAgICAgICAgICBicmVhawogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGZhaWxlZC5hcHBlbmQoeyJpZCI6IGl0WyJpZCJdLCAid2h5IjogImJsYWNrL05hTiBvdXRwdXQgdHdpY2UifSkKICAgICAgICByZXBvcnQoaW1hZ2VzPWRvbmUsIGZhaWxlZD1mYWlsZWQpCgoKZXJyID0gTm9uZQppZiBtb2RlbCA9PSAiemltYWdlIjoKICAgIHRyeToKICAgICAgICBnZW5femltYWdlKFBBUkFNU1siaXRlbXMiXSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICAgICAgICAgICAgICAgICAgICAgICMgT09NIG9yIGJyb2tlbiBvdXRwdXQgb24gdGhpcyBHUFUg4oaSIGxpZ2h0IG1vZGVsLCBzYW1lIGl0ZW1zCiAgICAgICAgZXJyID0gZiJ7dHlwZShlKS5fX25hbWVfX306IHtzdHIoZSlbOjMwMF19IgogICAgaWYgZXJyOiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBvdXRzaWRlIHRoZSBleGNlcHQgYmxvY2sgc28gdGhlIFotSW1hZ2UgcGlwZWxpbmUgaXMgZnJlZWQKICAgICAgICBpbXBvcnQgZ2M7IGdjLmNvbGxlY3QoKTsgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCiAgICAgICAgcmVwb3J0KHppbWFnZV9mYWlsZWQ9ZXJyLCBmYWxsYmFjaz0ic2R4bCIpCiAgICAgICAgUEFSQU1TWyJpdGVtcyJdID0gW2l0IGZvciBpdCBpbiBQQVJBTVNbIml0ZW1zIl0gaWYgbm90IChPVVQgLyBmIntpdFsnaWQnXX0ucG5nIikuZXhpc3RzKCldCiAgICAgICAgUEFSQU1TWyJtb2RlbCJdID0gInNkeGwiICAgICAgICAgICAgICAgICAjIHBlci1HUFUgY2hpbGRyZW4gcmUtcnVuIHRoaXMgZmlsZTogc2VuZCB0aGVtIHN0cmFpZ2h0IHRvIFNEWEwKICAgICAgICBnZW5fc2R4bChQQVJBTVNbIml0ZW1zIl0pCmVsc2U6CiAgICBnZW5fc2R4bChQQVJBTVNbIml0ZW1zIl0pCg==\'\nexec(base64.b64decode(_COMMON_B64).decode())\n\n\ndef _env_check():  # did Kaggle grant what the job asked for? (GPU + internet need a phone-verified account)\n    import shutil as _sh, socket as _so, subprocess as _sp\n    gpu = bool(_sh.which("nvidia-smi")) and _sp.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0\n    try:\n        _so.getaddrinfo("huggingface.co", 443); net = True\n    except OSError:\n        net = False\n    report(gpu_granted=gpu, internet_granted=net)\n    if PARAMS.get("_need_gpu") and not gpu:\n        report(error=("Kaggle gave this session no GPU: this accelerator is busy/unavailable or another GPU session "\n                      "is running (free accounts get one at a time); retry later.") if net else\n                     ("Kaggle gave this session no GPU and no internet: the account needs phone verification "\n                      "(kaggle.com > Settings > Phone verification)."), gpu_unavailable_reason="busy" if net else "unverified")\n        raise SystemExit(3)\n    if PARAMS.get("_need_internet") and not net:\n        report(error="Kaggle gave this session no internet (phone verification needed, or internet off).")\n        raise SystemExit(3)\n\n\n_env_check()\ntry:\n    import torch as _t\n    if _t.cuda.is_available():\n        report(gpu=gpu_info())\nexcept Exception:\n    pass\ntry:\n    exec(compile(base64.b64decode(_JOB_B64).decode(), "job.py", "exec"))\nexcept _Done:\n    pass\nif json.loads((OUT / "report.json").read_text()).get("error"):\n    raise SystemExit(4)\n'
open('/content/vs_job.py', 'w').write(JOB)
import subprocess, sys
rc = subprocess.run([sys.executable, '/content/vs_job.py']).returncode
print('job exit code', rc, '(0 = ok; see report.json in the zip either way)')


In [ ]:
# 4. pack the results and download them
import json, pathlib, shutil
out = pathlib.Path('/kaggle/working/out')
rep = out / 'report.json'
if rep.exists():
    d = json.loads(rep.read_text()); d['provider'] = 'colab'; rep.write_text(json.dumps(d, indent=2))
    print(json.dumps(d, indent=1)[:3000])
zp = shutil.make_archive('/content/vs-result-sdxl', 'zip', out)
from google.colab import files
files.download(zp)


In [ ]:
# 5. release the GPU (run after the download has started)
from google.colab import runtime
runtime.unassign()
